In [1]:
import os
import re
import json
from datetime import datetime
import random
import pandas as pd
from dotenv import load_dotenv
load_dotenv()

True

# 1. Test Dataset Generation from Documents

In [2]:
from evaluation.eval import prepare_testset_documents, generate_qa_dataset, generate_rag_responses

/Volumes/LaCie/Projects_portfolio/IntelliQA/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Volumes/LaCie/Projects_portfolio/IntelliQA/venv/lib/python3.11/site-packages/tika/__init__.py:20: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__('pkg_resources').declare_namespace(__name__)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 687.85it/s]


In [ ]:
docs = prepare_testset_documents("eval_data")

In [3]:
from rag_pipeline.query_engine import vectorstore

In [ ]:
vectorstore_db = vectorstore(persist_directory="/Volumes/LaCie/Projects_portfolio/IntelliQA/chroma_db", 
                                     documents=docs)

# 2. Synthetic QA Dataset Generation

In [9]:
# imports for synthetic QA Dataset Generation
from ragas import RunConfig
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_ollama import ChatOllama # ollama can run locally
from langchain_huggingface import HuggingFaceEmbeddings

In [ ]:
# csv files are temporarily filtered out because they are large and dominate the dataset
gen_docs = [file for file in docs if not file.metadata['filename'].endswith('.csv')]

# subsample documents to control cost and runtime during synthetic QA generation
random_docs = random.sample(gen_docs, 200)

run_config = RunConfig(max_workers=2, timeout=180)

In [ ]:
generator_llm = LangchainLLMWrapper(
                ChatOllama(
                    model="qwen2.5",
                    temperature=0,
                    # model_kwargs={"response_format":{"type":"json_object"}}
                    format="json"
                    )
                )

generator_embeddings = LangchainEmbeddingsWrapper(
                HuggingFaceEmbeddings(
                    model_name="sentence-transformers/all-MiniLM-L6-v2"
                    )
                )           

In [ ]:
testset = generate_qa_dataset(
                                random_docs, generator_llm, generator_embeddings,  
                                run_config=run_config,
                                testset_size=100
                            )

In [ ]:
df = testset.to_pandas()

In [ ]:
testset.to_pandas().to_json("test_data/rag_evaluation_testset.json", orient="records", indent=2)

# 3. RAG Pipeline Execution

In [4]:
df = pd.read_json("test_data/rag_evaluation_testset.json")

In [5]:
vectorstore_db = vectorstore('chroma_db')

Loading existing vector space


In [ ]:
generate_rag_responses(df[:32], vectorstore_db, session_id="test_session",k=8)

Row 1 processed and saved.
Row 2 processed and saved.
Row 3 processed and saved.
Row 4 processed and saved.
Row 5 processed and saved.
Row 6 processed and saved.
Row 7 processed and saved.
Row 8 processed and saved.
Row 9 processed and saved.
Row 10 processed and saved.
Row 11 processed and saved.
Row 12 processed and saved.
Row 13 processed and saved.
Row 14 processed and saved.
Row 15 processed and saved.
Row 16 processed and saved.
Row 17 processed and saved.
Row 18 processed and saved.
Row 19 processed and saved.
Row 20 processed and saved.
Row 21 processed and saved.
Row 22 processed and saved.
Row 23 processed and saved.
Row 24 processed and saved.
Row 25 processed and saved.
Row 26 processed and saved.
Row 27 processed and saved.
Row 28 processed and saved.
Row 29 processed and saved.
Row 30 processed and saved.
Row 31 processed and saved.
Row 32 processed and saved.


In [22]:
with open("test_data/rag_results.jsonl", "r") as f:
    rag_results = [json.loads(line) for line in f]

In [ ]:
# print(rag_results[-1]['user_input'])
print(df.iloc[31].user_input)

# 4. RAG Evaluation

In [7]:
from ragas import EvaluationDataset, evaluate
from ragas.metrics import (
    LLMContextPrecisionWithReference,
    LLMContextRecall,
    Faithfulness,
    ResponseRelevancy,
)
from langchain_google_genai import ChatGoogleGenerativeAI

In [10]:
evaluator_llm = LangchainLLMWrapper(
                    ChatGoogleGenerativeAI(
                        model="gemini-flash-lite-latest",
                        temperature=0,
                        # model_kwargs={"response_format": {"type": "json_object"}},
                    )
                )
evaluator_embeddings = LangchainEmbeddingsWrapper(
        HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
        )

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 412.57it/s]


In [23]:
metrics = [
    LLMContextPrecisionWithReference(llm=evaluator_llm),
    LLMContextRecall(llm=evaluator_llm),
    Faithfulness(llm=evaluator_llm),
    ResponseRelevancy(llm=evaluator_llm, embeddings=evaluator_embeddings),
]

run_config = RunConfig(max_workers=2, timeout=180)

In [ ]:
result = evaluate(dataset=EvaluationDataset.from_list(rag_results), metrics=metrics, 
                   run_config=run_config, raise_exceptions=True)

In [ ]:
result_df = result.to_pandas()

In [ ]:
result_df.to_csv("evaluation/rag_eval_v1.csv", index=False)